# Install dependencies

In [1]:
!pip install transformers datasets sentencepiece accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 851.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import numpy as np
from datasets import Dataset
from transformers import MT5Tokenizer

# Prepare the input dataset

In [15]:
# Carga el dataset
df = pd.read_csv('idealista_test.csv')

In [17]:
# Función para convertir una fila del DataFrame en una frase tipo input_text
def generar_input_text(row):
    extras = []
    for campo in ['terraza', 'garaje', 'ascensor', 'jardin', 'piscina', 'aire_acondicionado']:
        if row[campo] == True:
            extras.append(campo.replace('_', ' '))
    extras_texto = ", " + ", ".join(extras) if extras else ""

    return (
        f"{row['tipo']} de {row['metros']} m² en {row['zona']} con "
        f"{row['habitaciones']} habitaciones, {row['baños']} baños{extras_texto}"
    )

In [18]:
# Generar las nuevas columnas
df['input_text'] = df.apply(generar_input_text, axis=1)
df['target_text'] = df['descripcion']

In [19]:
df[['input_text', 'target_text']].to_csv("train_descripciones.csv", index=False)

# Cargar y tokenizar los datos

In [20]:
# Cargar y convertir a Dataset
df = pd.read_csv("train_descripciones.csv")
dataset = Dataset.from_pandas(df)

# Cargar tokenizer de mt5-small
tokenizer = MT5Tokenizer.from_pretrained("google/mt5-small")

# Preprocesamiento
def preprocess(example):
    model_input = tokenizer(example["input_text"], padding="max_length", truncation=True, max_length=128)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(example["target_text"], padding="max_length", truncation=True, max_length=128)
    model_input["labels"] = labels["input_ids"]
    return model_input

tokenized_dataset = dataset.map(preprocess, remove_columns=dataset.column_names)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.


Map:   0%|          | 0/59 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


# Entrenar el modelo

In [ ]:
from transformers import MT5ForConditionalGeneration, TrainingArguments, Trainer

model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")

training_args = TrainingArguments(
    output_dir="./mt5-descripciones",
    per_device_train_batch_size=4,
    num_train_epochs=5,
    logging_steps=100,
    save_total_limit=2,
    fp16=True,  # Activa si tienes GPU con soporte
    # The 'evaluation_strategy' argument was introduced in a later version.
    # For older versions, you might need to remove it or use a different approach
    # for evaluation. For example, you can manually evaluate the model after training.
    # evaluation_strategy="no",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

trainer.train()

<ipython-input-21-10022a056ef7>:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


# Guardar el modelo

In [11]:
model.save_pretrained("mt5-descripciones-v1")
tokenizer.save_pretrained("mt5-descripciones-v1")

('mt5-descripciones-v1/tokenizer_config.json',
 'mt5-descripciones-v1/special_tokens_map.json',
 'mt5-descripciones-v1/spiece.model',
 'mt5-descripciones-v1/added_tokens.json')

# Generar una descripción

In [14]:
def generar_descripcion(input_text):
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=128)
    outputs = model.generate(**inputs, max_length=128, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Ejemplo
descripcion = generar_descripcion("piso de 95 m² en Centre con 3 habitaciones, 2 baños, terraza y ascensor")
print(descripcion)

zuwaīcijবানeteiemantaingpellingchaussडाय <extra_id_24>измеy <extra_id_47>USEDలిక- <extra_id_48>
